In [1]:
import torch
from torch import nn
from d2l import torch as d2l

seed=42
torch.manual_seed(seed)
class Reshape(torch.nn.Module):
    def forward(self,x):
        return x.reshape(-1,1,28,28)

net=torch.nn.Sequential(
    Reshape(),nn.Conv2d(1,32,kernel_size=5,padding=2),nn.BatchNorm2d(32),nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32,64,kernel_size=5),nn.ReLU(),
    nn.MaxPool2d(2),nn.BatchNorm2d(64),nn.Flatten(),
    nn.Linear(64*5*5,128),nn.ReLU(),
    nn.Linear(128,84),nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(84,10)  
)

In [2]:
def evaluate_accuracy_gpu(net,data_iter,device=None):
    if isinstance(net,nn.Module):
        net.eval()
        if device is None:
            device=next(net.parameters()).device

    correct=0
    total=0

    with torch.inference_mode():
        for X,y in data_iter:
            if isinstance(X,list):
                X=[x.to(device) for x in X]
            else:
                X=X.to(device)

            y=y.to(device)
            y_hat=net(X)

            pred=y_hat.argmax(dim=1)
            correct+=(pred==y).sum().item()
            total+=y.numel()
    return correct/total

In [3]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.RandomRotation(10),
])


class MyDataset(torch.utils.data.Dataset):
    def __init__(self,X,y,transform=None):
        self.X=X
        self.y=y
        self.transform=transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self,index):
        img=self.X[index]
        label=self.y[index]

        if self.transform:
            img=self.transform(img)

        return img,label

In [4]:
import pandas as pd
from torch.utils.data import TensorDataset,random_split,DataLoader
#读取训练集和测试集
train_data=pd.read_csv("train.csv")
test_data=pd.read_csv("test.csv")

y_series=train_data["label"]#读取标签列
x_df=train_data.drop(columns=["label"])#去除标签列

X=torch.tensor(x_df.to_numpy(),dtype=torch.float32)
y=torch.tensor(y_series.to_numpy(),dtype=torch.long)
X_test=torch.tensor(test_data.to_numpy(),dtype=torch.float32)

X=X.reshape(-1,1,28,28)
X=X/255.0#归一化--原始像素在0-255区间内
X_test=X_test.reshape(-1,1,28,28)
X_test=X_test/255.0

dataset=MyDataset(
    X,
    y,
    transform=transforms.RandomRotation(10)
)

#从完整的训练集中拆分出训练集和验证集--比例为9：1
# train_dataset,val_dataset=random_split(
#     dataset,
#     [37800,4200],
#     #固定随机种子，让 random_split() 每次随机拆分出来的训练集和验证集都一样
#     generator=torch.Generator().manual_seed(seed)
# )
#加载训练集
train_iter=DataLoader(
    dataset,
    batch_size=256,
    shuffle=True
)
#加载验证集
# val_iter=DataLoader(
#     val_dataset,
#     batch_size=256,
#     shuffle=False
# )
#加载测试集
test_iter=DataLoader(
    X_test,
    batch_size=256,
    shuffle=False
)

In [5]:
#定义初始化函数
def init_weights(m):
    if type(m)==nn.Linear or type(m)==nn.Conv2d:
        nn.init.kaiming_uniform_(m.weight)
#定义训练函数
def train_one_epoch(net,train_iter,optimizer,loss_fn,device):
    loss_sum=0.0
    correct=0
    total=0
    for X,y in train_iter:
        
        optimizer.zero_grad()
        X,y=X.to(device),y.to(device)

        y_hat=net(X)
        l=loss_fn(y_hat,y)
        l.backward()
        optimizer.step()

        pred=y_hat.argmax(dim=1)
        loss_sum+=l.item() * X.shape[0]
        correct+=(pred==y).sum().item()
        total+=y.numel()

        train_l=loss_sum/total
        train_acc=correct/total

    return train_l,train_acc

net.apply(init_weights)
#启用GPU
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('training on',device)
net.to(device)
#定义参数
lr,num_epochs=0.000550901,40#定义学习率和训练轮数
batch_size=256
optimizer=torch.optim.Adam(net.parameters(),lr=lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)
loss=nn.CrossEntropyLoss()
#开始训练
best_acc=0#记录最高验证准确率

for epoch in range(num_epochs):
    net.train()
    scheduler.step()
    train_l,train_acc=train_one_epoch(net,train_iter,optimizer,loss,device)
    # val_acc=evaluate_accuracy_gpu(net,val_iter)

    #如果当前模型验证集更好，就保存
    # if val_acc>best_acc:
    #     best_acc=val_acc
    #     torch.save(net.state_dict(),"best.pt")
    #     print("保存最佳模型,val_acc:",best_acc)

    print(epoch + 1, train_l, train_acc)

training on cuda


C:\Users\34331\AppData\Local\Temp\ipykernel_10028\3844670091.py:49: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


1 0.47017773184322176 0.8550476190476191
2 0.1432089472143423 0.959047619047619
3 0.0966563157637914 0.9727857142857143
4 0.0835112209433601 0.9772380952380952
5 0.06551160877659207 0.9813095238095239
6 0.0536430911620458 0.9845714285714285
7 0.04696381119035539 0.9865
8 0.042369262224506765 0.9880714285714286
9 0.03581105099831309 0.9900476190476191
10 0.029710999236220406 0.9910714285714286
11 0.03072637372490551 0.9909761904761905
12 0.024649994278947512 0.9927857142857143
13 0.02214356715054739 0.9935
14 0.02141608364809127 0.9934047619047619
15 0.019597713691138087 0.9943809523809524
16 0.017817077992040485 0.9950714285714286
17 0.01698747247279001 0.9950952380952381
18 0.017075627118515384 0.9952619047619048
19 0.016091689836034286 0.9953571428571428
20 0.015838669830534075 0.9952142857142857
21 0.016599196780206903 0.994904761904762
22 0.016536753365681285 0.9952857142857143
23 0.01639779084460627 0.9953333333333333
24 0.01630499962276025 0.9948333333333333
25 0.0172072665379604

In [6]:
#加载最佳模型
net.load_state_dict(torch.load("best.pt"))
#测试模型
net.eval()

with torch.inference_mode():
    predictions=[]
    for X in test_iter:
        X=X.to(device)
        y_hat=net(X)
        pred=y_hat.argmax(dim=1)
        #把pred收集起来
        predictions.extend(pred.cpu().tolist())
submission = pd.read_csv("sample_submission.csv")

submission["Label"] = predictions


In [7]:
submission.to_csv("submission.csv", index=False)